# 6주차. WebUI와 Sub-Agent 통합 데모

## 0. 목표

이번 주에는 나나/카나 통합 흐름을 WebUI에서 실행하고, trace를 보며 설명 가능한 최종 데모로 정리한다.

성취 기준:

- 최종 UI에서 입력, 선택 agent, 내부 tool, 최종 답변을 연결해 설명할 수 있다.
- `run_practice_suite` 결과로 golden scenario 통과 여부를 검증할 수 있다.
- 실패 가능성과 사람이 검토해야 할 값을 발표에 포함할 수 있다.


## 1. 준비

로컬 `Jupyter Notebook` 또는 `JupyterLab`에서 repo 루트를 기준으로 실행한다.

모든 실습은 실제 OpenAI API를 호출한다. API 키와 모델 설정은 repo 루트의 `.env`에서 읽는다.

처음 읽는 순서: `docs/orientation.md` → 이 노트북 → `week06.py`의 `# TODO 문제`를 먼저 풀고 바로 아래 `# 모범 답안`과 비교한 뒤 Gradio 데모를 실행한다.


In [ ]:
# 흐름: repo 설정과 공통 helper를 준비한다.
# 최종 데모는 답변만 보는 것이 아니라 선택 agent, payload, trace를 함께 본다.
import json
import os
import sys
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() or (candidate / ".env.example").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. repo 안에서 노트북을 실행하세요.")


REPO_ROOT = find_repo_root(Path.cwd())

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv(REPO_ROOT / ".env", override=True)

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("repo 루트 .env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")


def make_model(max_tokens: int = 500) -> ChatOpenAI:
    return ChatOpenAI(model=OPENAI_MODEL, temperature=0, max_completion_tokens=max_tokens)


def show_json(value: Any) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    trace = []
    for message in agent_result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace


## 2. 개념

오늘의 큰 그림: 최종 데모는 단순히 화면이 작동하는지 보는 것이 아니라, supervisor가 호출한 sub-agent tool, sub-agent 내부 tool trace, final answer를 한 번에 확인하는 설명 가능한 사용자 경험이다.

반드시 이해할 것:

- `mode`는 supervisor가 어떤 agent 후보를 사용할지 제한한다.
- `delegate_payload` 안에 sub-agent의 답변과 내부 trace가 들어 있다.
- 최종 발표는 입력, 선택 agent, 내부 tool, payload, 실패 가능성을 함께 다룬다.

지금은 몰라도 되는 것:

- Gradio UI를 production 서비스로 배포하는 방법
- 복잡한 상태 관리나 사용자 인증
- 장기 운영을 위한 모니터링 설계

막혔을 때 볼 trace: supervisor trace, `delegate_payload.trace`, `selected_agent`, `inner_tool_names`, `passed`.


## 3. 기본 개념 실습

가장 작은 흐름은 나나/카나 sub-agent와 supervisor를 한 노트북에서 함께 준비하는 것이다.


In [ ]:
# 흐름: 최종 데모에서 쓸 Nana/Kana sub-agent와 supervisor 위임 tool을 만든다.
# Nana는 개인 일정/메모, Kana는 그룹 일정 조율 역할로 분리한다.
@tool("personal_create_schedule", description="개인 일정을 생성한다.")
def personal_create_schedule(title: str, date: str, start_time: str) -> str:
    """Create a personal schedule."""
    return json.dumps({"ok": True, "schedule": {"title": title, "date": date, "start_time": start_time}}, ensure_ascii=False)

@tool("memory_save", description="사용자 메모를 저장한다.")
def memory_save(title: str, content: str) -> str:
    """Save a user memory."""
    return json.dumps({"ok": True, "memory": {"title": title, "content": content}}, ensure_ascii=False)

@tool("group_confirm_slot", description="그룹 일정 시간을 확정한다.")
def group_confirm_slot(topic: str, selected_slot: str, members: list[str], reason: str) -> str:
    """Confirm a group schedule slot."""
    return json.dumps({"ok": True, "topic": topic, "selected_slot": selected_slot, "members": members, "reason": reason}, ensure_ascii=False)
nana_agent = create_agent(
    model=make_model(700),
    tools=[personal_create_schedule, memory_save],
    system_prompt="너는 나나다. 오늘은 2026-04-23이다. 상대 날짜는 이 날짜 기준으로 YYYY-MM-DD로 바꾼다. 개인 일정이나 메모 요청에 필요한 도구를 호출한다.",
)

kana_agent = create_agent(
    model=make_model(800),
    tools=[group_confirm_slot],
    system_prompt="너는 카나다. 그룹 응답에서 공통 가능 시간을 찾으면 group_confirm_slot 도구를 호출한다.",
)

@tool("nana_agent", description="개인 일정이나 메모 요청을 나나 sub-agent에게 위임한다.")
def delegate_to_nana(request: str) -> str:
    """Delegate a personal request to Nana sub-agent."""
    agent_result = nana_agent.invoke({"messages": [{"role": "user", "content": request}]})
    return json.dumps({"agent": "nana", "answer": final_text(agent_result), "trace": extract_tool_trace(agent_result)}, ensure_ascii=False)

@tool("kana_agent", description="그룹 일정 조율 요청과 멤버 응답을 카나 sub-agent에게 위임한다.")
def delegate_to_kana(request: str, member_replies: str) -> str:
    """Delegate a group coordination request to Kana sub-agent."""
    message = f"요청: {request}\n그룹 응답:\n{member_replies}"
    agent_result = kana_agent.invoke({"messages": [{"role": "user", "content": message}]})
    return json.dumps({"agent": "kana", "answer": final_text(agent_result), "trace": extract_tool_trace(agent_result)}, ensure_ascii=False)

def make_supervisor(mode: str = "auto"):
    if mode == "personal":
        tools = [delegate_to_nana]
        prompt = "너는 카나메이트 supervisor다. 사용자가 personal 모드를 선택했으므로 nana_agent tool을 호출한다."
    elif mode == "group":
        tools = [delegate_to_kana]
        prompt = "너는 카나메이트 supervisor다. 사용자가 group 모드를 선택했으므로 kana_agent tool을 호출한다."
    else:
        tools = [delegate_to_nana, delegate_to_kana]
        prompt = "너는 카나메이트 supervisor다. 개인 일정/메모 요청은 nana_agent tool을 호출하고, 그룹 일정 조율 요청은 kana_agent tool을 호출한다. 직접 처리하지 말고 반드시 적절한 sub-agent tool을 호출한다."
    return create_agent(model=make_model(1000), tools=tools, system_prompt=prompt)


def delegated_agent_from_trace(agent_result: dict[str, Any]) -> str:
    for event in extract_tool_trace(agent_result):
        if event.get("event") == "tool_call" and event.get("tool_name") in {"nana_agent", "kana_agent"}:
            return event["tool_name"].replace("_agent", "")
    return "unknown"


def delegated_payload_from_trace(agent_result: dict[str, Any]) -> dict[str, Any]:
    for event in extract_tool_trace(agent_result):
        if event.get("event") == "tool_result" and event.get("tool_name") in {"nana_agent", "kana_agent"}:
            return json.loads(event["content"])
    return {}


## 4. 카나메이트 확장 예제

준비한 supervisor를 `run_live_flow`로 감싼다. 이 함수는 WebUI와 최종 Golden Scenario가 함께 사용할 선택 agent, 최종 답변, trace, delegate payload를 반환한다.


In [ ]:
# 흐름: live flow 한 번을 실행하는 함수와 golden scenario suite를 정의한다.
# UI와 자동 점검이 같은 runner를 쓰면 발표 흐름과 검증 흐름이 일치한다.
def run_live_flow(student_request: str, member_replies: str = "", mode: str = "auto") -> dict[str, Any]:
    supervisor = make_supervisor(mode)
    content = f"요청: {student_request}\n그룹 응답:\n{member_replies}" if mode in {"auto", "group"} else student_request
    supervisor_result = supervisor.invoke({"messages": [{"role": "user", "content": content}]})
    return {"selected_agent": delegated_agent_from_trace(supervisor_result), "answer": final_text(supervisor_result), "trace": extract_tool_trace(supervisor_result), "delegate_payload": delegated_payload_from_trace(supervisor_result)}

student_request = "팀 멤버들과 발표 리허설 시간을 조율해줘"
member_replies = "민수: 2026-04-24 15:00 가능\n지아: 2026-04-24 15:00 가능"
run_result = run_live_flow(student_request, member_replies, mode="auto")
show_json(run_result)


## 5. 확장 예제 실행

`run_live_flow`를 새 입력으로 실행한다. 수강생은 `mini_request`와 `mode`를 바꿔 WebUI에서 확인할 시나리오를 먼저 노트북에서 점검한다.


In [ ]:
# 흐름: 개인 메모 저장 미니 케이스를 실행해 최종 결과를 trace로 설명해본다.
# selected_agent와 mini_inner_tools가 기대와 맞는지 확인한다.
mini_request = "프로젝트 발표 장소는 3층 세미나실이라고 메모해줘"
mini_result = run_live_flow(mini_request, mode="personal")
mini_inner_tools = [
    event["tool_name"]
    for event in mini_result["delegate_payload"].get("trace", [])
    if event.get("event") == "tool_call"
]

show_json(mini_result)

assert mini_result["selected_agent"] == "nana"
assert mini_result["answer"]
assert any(event["event"] == "tool_call" and event["tool_name"] == "nana_agent" for event in mini_result["trace"])
assert "memory_save" in mini_inner_tools
print("6주차 확장 예제 실행 통과")


## 6. 회고

개념 확인 질문:

1. 최종 WebUI에서 `selected_agent`와 `delegate_payload`를 함께 보여주는 이유는 무엇인가?
2. `passed=True`를 만들기 위해 어떤 trace 값을 확인해야 하는가?
3. 이 데모에서 사람이 검토해야 할 값은 무엇이며, 자동 실행하면 위험한 부분은 무엇인가?

작은 응용 과제: KanaMate 개선 미니 프로젝트 아이디어 하나를 정하고, 입력, 선택 agent, 내부 tool, payload, 실패 가능성 순서로 발표 템플릿을 작성한다.
